# 🚀 zgw-posix: Enterprise S3-to-POSIX Demo
This notebook demonstrates a full data lifecycle: from S3 API ingestion to POSIX filesystem verification.

---

In [ ]:
# 1. Setup & Connection
!pip install -q boto3 pandas matplotlib seaborn pillow
import os, io, boto3, random, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from botocore.config import Config
from datetime import datetime

S3_ENDPOINT = os.environ.get('S3_ENDPOINT_URL', 'http://s3-posix')
AWS_ACCESS_KEY = "zippy"
AWS_SECRET_KEY = "zippy"

s3_client = boto3.client(
    's3', 
    endpoint_url=S3_ENDPOINT, 
    aws_access_key_id=AWS_ACCESS_KEY, 
    aws_secret_access_key=AWS_SECRET_KEY, 
    region_name='default' # Required for Ceph RGW
)
print(f"✅ Connected to: {S3_ENDPOINT}")

## 2. Bucket Management
First, we explicitly create our demo environment.

In [ ]:
TARGET_BUCKET = 'analytics-demo'
print(f"🔨 Creating bucket: {TARGET_BUCKET}...")
try:
    s3_client.create_bucket(Bucket=TARGET_BUCKET)
    print("✨ Bucket created successfully.")
except Exception as e: 
    print(f"ℹ️ {e}")

In [ ]:
print("📋 Listing all available S3 Buckets:")
response = s3_client.list_buckets()
for i, b in enumerate(response['Buckets'], 1):
    print(f"{i}. {b['Name']}")

## 3. Data Ingestion
We generate telemetry data and upload it using a 'pseudo-directory' path.

In [ ]:
df = pd.DataFrame({
    'sensor': [f'SNS-{i}' for i in range(100)],
    'cpu_load': [random.uniform(10, 90) for _ in range(100)],
    'mem_usage': [random.uniform(20, 80) for _ in range(100)]
})
csv_buf = io.StringIO()
df.to_csv(csv_buf, index=False)

OBJECT_KEY = 'raw/telemetry.csv'
s3_client.put_object(Bucket=TARGET_BUCKET, Key=OBJECT_KEY, Body=csv_buf.getvalue())
print(f"📤 Dataset uploaded to: s3://{TARGET_BUCKET}/{OBJECT_KEY}")

In [ ]:
print(f"🔍 Listing objects inside '{TARGET_BUCKET}':")
objs = s3_client.list_objects_v2(Bucket=TARGET_BUCKET)
for o in objs.get('Contents', []):
    print(f"  - {o['Key']} ({o['Size']} bytes)")

## 4. Visual Dashboard
Processing the S3 data to create a real-time visualization.

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 5))
sns.histplot(df['cpu_load'], kde=True, color="#3498db")
plt.title("CPU Load Distribution (Source: s3-posix)", fontsize=16)
plt.show()

## 5. The 'Killer Feature': POSIX Transparency
**Crucial Step:** We verify the file on disk. Note that zgw-posix encodes `/` as `%2F` on the filesystem.

In [ ]:
# The S3 key 'raw/telemetry.csv' becomes 'raw%2Ftelemetry.csv' on disk
posix_filename = OBJECT_KEY.replace('/', '%2F')
posix_path = f"/var/lib/ceph/rgw_posix_driver/{TARGET_BUCKET}/{posix_filename}"

print(f"🔎 Investigating Filesystem Path: {posix_path}\n")

if os.path.exists(posix_path):
    print("🚀 PROOF: The S3 object is stored as a standard file!")
    !ls -lh {posix_path}
else:
    print("❌ Path not found. Debugging directory contents:")
    !ls -R /var/lib/ceph/rgw_posix_driver/

## 6. Final Report Generation
Generating a summary report and archiving it back to S3.

In [ ]:
report_buf = io.BytesIO()
plt.figure(figsize=(6, 3))
plt.text(0.5, 0.5, f"DEMO COMPLETE\nStorage Path: {posix_path}\nObjects: {len(df)} rows", 
         ha='center', va='center', size=12, bbox=dict(boxstyle='round', facecolor='lightgreen'))
plt.axis('off')
plt.savefig(report_buf, format='png')

s3_client.put_object(Bucket=TARGET_BUCKET, Key='reports/final_summary.png', Body=report_buf.getvalue())
print("📊 Final report archived to S3. Demo finished.")